# Understanding Feature Relationships Through EDA and Relation Networks

When you have a dataset with many features and one outcome you care about, two natural questions arise:

1. **Which features matter?** — Which ones are most related to the outcome?
2. **How do features relate to each other?** — Do some features move together? Do they form groups?

This notebook walks through both questions using three very different datasets. For each one, we start with a simple visualization that shows the clearest pattern, then build up to a **relation network** — a single picture that captures both feature→outcome and feature↔feature relationships at once.

| Dataset | Target | What makes it interesting |
|---|---|---|
| World Happiness | Happiness Score (continuous) | Strong linear trends visible in scatter plots |
| Lung Cancer | Cancer Yes/No (binary) | One feature group (smoking) dominates |
| Network Intrusion (NSL-KDD) | Attack / Normal (binary) | 38 numeric features in 4 natural domain groups |

**Tool used throughout**: Spearman correlation (ρ) — a non-parametric measure of how strongly two variables move together. No model training needed; this is pure exploratory data analysis (EDA).

## Setup

Run these cells first. They install any missing packages, download the datasets, and define the helper functions we'll reuse across all three analyses.

> **Google Colab**: everything downloads automatically — just run the cells in order.  
> **Local machine**: the data files will be downloaded into subfolders alongside the notebook.

In [ ]:
# ── Install extra packages (pre-installed on Colab except xgboost/lightgbm) ──
%pip install -q xgboost lightgbm

# ── Standard data science libraries ──
import itertools          # for generating all feature pairs
import re                 # for cleaning text labels
import textwrap           # for wrapping long labels in plots
import urllib.request     # for downloading datasets
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx     # for building and drawing network graphs
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import spearmanr  # Spearman correlation function

warnings.filterwarnings('ignore')

# Render figures as SVG (vector) instead of PNG — sharper at any zoom level
%config InlineBackend.figure_formats = ['svg']

# Global plot style: clean grid, readable font sizes
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlepad'] = 14

# ── Data download helper ──
DATA_BASE_URL = 'https://raw.githubusercontent.com/arhsis/UM-CISC7107-assignment2/main/'

def download_if_missing(relative_path):
    """Download a dataset file from GitHub if it doesn't exist locally.
    
    relative_path: e.g. 'happiness/data_cleaned_compact.csv'
    Downloads from DATA_BASE_URL + relative_path into the same local path.
    """
    local = Path(relative_path)
    if local.exists():
        return  # already have it
    local.parent.mkdir(parents=True, exist_ok=True)
    url = DATA_BASE_URL + relative_path.replace('+', '%2B')
    print(f'Downloading {relative_path} …')
    urllib.request.urlretrieve(url, str(local))
    print('Done.')

# Download all datasets upfront
download_if_missing('happiness/data_cleaned_compact.csv')
download_if_missing('lung_cancer_mixed_types_attributes/LungCancer_cancerYesNo-grouped.csv')
download_if_missing('nsl_kdd/KDDTrain+.txt')

print('All datasets ready.')

In [ ]:
# ── Shared helper functions ──
# These are reused across all three datasets.


def wrap_label(text, width=20):
    """Wrap long text into multiple lines so it fits inside a plot node or axis label.
    
    Example: "GDP per Capita (constant 2015 US$)" → "GDP per Capita\n(constant 2015 US$)"
    """
    return textwrap.fill(str(text), width=width, break_long_words=False, break_on_hyphens=False)


def compute_spearman(X, y, min_n=25):
    """Measure how strongly each feature correlates with the outcome.
    
    How it works:
      For each feature column in X, compute Spearman ρ against y.
      Spearman ρ ranges from -1 to +1:
        +1 = perfect positive relationship (feature goes up → outcome goes up)
        -1 = perfect negative relationship (feature goes up → outcome goes down)
         0 = no relationship
    
    We skip features with too few valid rows (< min_n) or no variation.
    Returns a DataFrame sorted by |ρ| (strongest correlations first).
    """
    rows = []
    for col in X.columns:
        # Only use rows where both feature and outcome are non-null
        valid = X[col].notna() & y.notna()
        if valid.sum() < min_n or X.loc[valid, col].nunique() < 2:
            continue
        rho, p = spearmanr(X.loc[valid, col], y[valid])
        if not np.isnan(rho):
            rows.append({'feature': col, 'spearman_rho': float(rho),
                         'abs_rho': abs(float(rho)), 'p_value': float(p)})
    return pd.DataFrame(rows).sort_values('abs_rho', ascending=False).reset_index(drop=True)


def build_signed_positions(
    center_node, positive_features, negative_features,
    strength_lookup, group_lookup,
    base_radius=5.0, right_span=(-1.15, 1.15), left_span=(2.0, 4.28),
):
    """Arrange features in a circle around the outcome node, split by sign.
    
    How it works:
      - The outcome (center_node) sits at the origin (0, 0).
      - Features with POSITIVE correlation go on the RIGHT side.
      - Features with NEGATIVE correlation go on the LEFT side.
      - Within each side, features are spread evenly along an arc.
      - Stronger correlations are placed slightly closer to the center.
      - Features are sorted by group first, so same-group nodes sit together.
    
    Returns a dict {node_name: (x, y)} for all nodes.
    """
    pos = {center_node: np.array([0.0, 0.0])}

    def place(features, span):
        if not features:
            return
        # Sort by group (so same-colored nodes cluster), then by strength
        ordered = sorted(features, key=lambda f: (str(group_lookup(f)), -strength_lookup[f], str(f)))
        angles = np.linspace(span[0], span[1], len(ordered))
        for angle, feat in zip(angles, ordered):
            # Stronger features → slightly smaller radius → closer to center
            strength = min(max(float(strength_lookup[feat]), 0.0), 1.0)
            r = base_radius + 0.45 * (1 - strength)
            pos[feat] = np.array([r * np.cos(angle), r * np.sin(angle)])

    place(positive_features, right_span)
    place(negative_features, left_span)
    return pos


def draw_relation_network(
    target_col, outcome_name, focus_features, focus_assoc, label_fn, group_fn,
    group_colors, X_data, ff_threshold=0.30,
    pos_label='Positive', neg_label='Negative',
    sign_flip=False, title='Relation Network',
):
    """Draw the full outcome-centered relation network.
    
    This is the "punchline" visualization. It shows:
      - Feature → Outcome edges (solid lines): how each feature relates to the outcome.
        Thicker = stronger correlation. Color = direction (green/red).
      - Feature ↔ Feature edges (dashed lines): how features relate to each other.
        Only drawn when |ρ| > ff_threshold (to avoid clutter).
      - Node colors: which domain group each feature belongs to.
      - Node sizes: larger = stronger correlation with outcome.
      - Layout: positive features on the right, negative on the left.
    
    sign_flip=True inverts the color meaning (useful when positive ρ = bad outcome,
    e.g. higher cancer risk or higher attack likelihood).
    """
    lookup = focus_assoc.set_index('feature')

    # ── Step 1: Build the graph ──
    # Add outcome as the center node
    G = nx.Graph()
    G.add_node(target_col, label=outcome_name, group='Outcome', outcome_strength=1.0)

    # Add each feature as a node, with an edge to the outcome
    for feat in focus_features:
        rho = float(lookup.loc[feat, 'spearman_rho'])
        G.add_node(feat, label=label_fn(feat), group=group_fn(feat), outcome_strength=abs(rho))
        G.add_edge(target_col, feat, relation='feature_outcome',
                   weight=abs(rho), sign=1 if rho >= 0 else -1, value=rho)

    # Add feature↔feature edges (only if correlation is strong enough)
    for f1, f2 in itertools.combinations(focus_features, 2):
        if f1 not in X_data.columns or f2 not in X_data.columns:
            continue
        pair = X_data[[f1, f2]].dropna()
        if len(pair) < 25:
            continue
        rho, _ = spearmanr(pair[f1], pair[f2])
        if pd.isna(rho) or abs(rho) < ff_threshold:
            continue
        G.add_edge(f1, f2, relation='feature_feature',
                   weight=abs(float(rho)), sign=1 if rho >= 0 else -1, value=float(rho))

    # ── Step 2: Compute layout positions ──
    pos_feats = sorted([f for f in focus_features if float(lookup.loc[f, 'spearman_rho']) >= 0],
                       key=lambda f: float(lookup.loc[f, 'abs_rho']), reverse=True)
    neg_feats = sorted([f for f in focus_features if float(lookup.loc[f, 'spearman_rho']) < 0],
                       key=lambda f: float(lookup.loc[f, 'abs_rho']), reverse=True)
    strength = {f: float(lookup.loc[f, 'abs_rho']) for f in focus_features}

    if neg_feats:
        positions = build_signed_positions(target_col, pos_feats, neg_feats, strength, group_fn)
    else:
        positions = build_signed_positions(target_col, pos_feats, [], strength, group_fn,
                                          right_span=(-2.2, 2.2))

    # Keep only the top 12 feature↔feature edges to avoid visual clutter
    ff_edges = sorted([(u, v, d) for u, v, d in G.edges(data=True)
                       if d.get('relation') == 'feature_feature'],
                      key=lambda e: e[2]['weight'], reverse=True)[:12]

    def edge_color(sign):
        if sign_flip:
            return '#C0392B' if sign >= 0 else '#2E8B57'
        return '#2E8B57' if sign >= 0 else '#C0392B'

    # ── Step 3: Draw everything ──
    fig, ax = plt.subplots(figsize=(14, 11))
    ax.set_title(title, fontsize=16, pad=16)

    # Draw feature↔feature edges first (behind, dashed, faint)
    for u, v, d in ff_edges:
        x0, y0 = positions[u]; x1, y1 = positions[v]
        ax.plot([x0, x1], [y0, y1], color=edge_color(d['sign']), alpha=0.20,
                linewidth=1 + 3.4 * d['weight'], linestyle='--', zorder=1)

    # Draw feature→outcome edges (solid, prominent)
    for u, v, d in G.edges(data=True):
        if d['relation'] != 'feature_outcome': continue
        x0, y0 = positions[u]; x1, y1 = positions[v]
        ax.plot([x0, x1], [y0, y1], color=edge_color(d['sign']), alpha=0.94,
                linewidth=2 + 7.0 * d['weight'], zorder=2)

    # Draw nodes (colored by group, sized by correlation strength)
    for group, color in group_colors.items():
        nodelist = [n for n, a in G.nodes(data=True) if a.get('group') == group]
        if not nodelist: continue
        sizes = [5200 if n == target_col else 1200 + 3600 * G.nodes[n]['outcome_strength'] for n in nodelist]
        nx.draw_networkx_nodes(G, positions, nodelist=nodelist, node_color=color,
                               node_size=sizes, edgecolors='black', linewidths=1.4, alpha=0.96, ax=ax)

    # Draw labels on top of nodes
    for node, (xc, yc) in positions.items():
        w = 18 if node != target_col else 16
        ax.text(xc, yc, wrap_label(G.nodes[node]['label'], width=w),
                fontsize=10, ha='center', va='center',
                fontweight='bold' if node == target_col else 'normal',
                bbox=dict(boxstyle='round,pad=0.32', fc='white', ec='#d0d0d0', lw=0.6, alpha=0.92), zorder=3)

    # Side labels showing which direction means what
    if neg_feats:
        ax.text(-7.2, 5.2, neg_label, color='#C0392B' if not sign_flip else '#2E8B57',
                fontsize=13, fontweight='bold')
    ax.text(4.3, 5.2, pos_label, color='#2E8B57' if not sign_flip else '#C0392B',
            fontsize=13, fontweight='bold')

    # Legends (inside the figure)
    pos_color = '#2E8B57' if not sign_flip else '#C0392B'
    neg_color = '#C0392B' if not sign_flip else '#2E8B57'
    edge_legend = ax.legend(
        handles=[
            Line2D([0], [0], color=pos_color, lw=4, label=pos_label),
            Line2D([0], [0], color=neg_color, lw=4, label=neg_label),
            Line2D([0], [0], color='gray', lw=3, linestyle='-', label='Feature → outcome'),
            Line2D([0], [0], color='gray', lw=3, linestyle='--', label='Feature ↔ feature'),
        ],
        loc='lower right', fontsize=9, frameon=True, title='Edge Meaning', title_fontsize=10,
    )
    ax.add_artist(edge_legend)

    ax.legend(
        handles=[Patch(facecolor=c, edgecolor='black', label=g) for g, c in group_colors.items()],
        loc='upper right', fontsize=9, frameon=True, title='Node Groups', title_fontsize=10,
    )

    ax.set_aspect('equal'); ax.axis('off')
    fig.tight_layout()
    plt.show()


def select_focus_features(corr_df, core_drivers, n_extra_pos=4, n_neg=6):
    """Choose which features to include in the relation network.
    
    We can't show all features (too cluttered), so we pick a focused subset:
      1. Start with the "core drivers" — features we know are important from domain knowledge.
      2. Add the next strongest positively-correlated features (n_extra_pos).
      3. Add the strongest negatively-correlated features (n_neg).
    
    This gives a balanced view: known important features + data-driven discoveries + both directions.
    """
    available_core = [c for c in core_drivers if c in corr_df['feature'].values]
    pos_cand = corr_df[corr_df['spearman_rho'] > 0].sort_values('spearman_rho', ascending=False)['feature'].tolist()
    neg_cand = corr_df[corr_df['spearman_rho'] < 0].sort_values('spearman_rho', ascending=True)['feature'].tolist()
    extra_pos = [f for f in pos_cand if f not in available_core][:n_extra_pos]
    neg_focus = neg_cand[:n_neg]
    focus = list(dict.fromkeys(available_core + extra_pos + neg_focus))
    return [f for f in focus if f in set(corr_df['feature'])]


print('Setup complete — all helper functions loaded.')

---
# Part 1: World Happiness Dataset

**Target**: Happiness Score (continuous, 0–8 scale)

This dataset contains country-level indicators — GDP, social support, life expectancy, freedom, generosity, corruption perception — alongside a happiness score from the World Happiness Report.

Because the target is continuous, **scatter plots** are the most natural way to see how each feature relates to happiness. We'll look at those first, then combine everything into a relation network.

### 1.1 Load Data

In [25]:
raw_hap = pd.read_csv('happiness/data_cleaned_compact.csv', nrows=3, header=0)
meta_row_hap = raw_hap.iloc[1]
df_hap = pd.read_csv('happiness/data_cleaned_compact.csv', skiprows=[1, 2])

hap_target = 'HAP.SCORE'
hap_id_cols = ['index', 'Country name']

hap_pretty = {
    'HAP.SCORE': 'Happiness Score',
    'EXP.GDP.PER.CAPITA': 'GDP per Capita',
    'EXP.SOC.SUPP': 'Social Support',
    'EXP.HEAL.LIFE': 'Healthy Life Expectancy',
    'EXP.FREE.LIFE': 'Freedom',
    'EXP.GEN': 'Generosity',
    'EXP.TRUST': 'Corruption Perception',
}

direct_map_hap = {
    'GDP per capita (constant 2015 US$)': 'GDP per Capita (WB)',
    'GDP (current US$)': 'GDP (Current US$)',
    'Life expectancy at birth, total (years)': 'Life Expectancy (WB)',
    'Mortality rate, adult, female (per 1,000 female adults)': 'Adult Female Mortality',
    'Mortality rate, adult, male (per 1,000 male adults)': 'Adult Male Mortality',
    'Birth rate, crude (per 1,000 people)': 'Birth Rate',
    'Fertility rate, total (births per woman)': 'Fertility Rate',
    'Agriculture, forestry, and fishing, value added (% of GDP)': 'Agriculture % of GDP',
    'Age dependency ratio, young (% of working-age population)': 'Young Dependency Ratio',
    'Age dependency ratio (% of working-age population)': 'Dependency Ratio',
}

for col in df_hap.columns:
    if col in hap_pretty or col in hap_id_cols:
        continue
    meta_val = str(meta_row_hap.get(col, ''))
    m = re.search(r'Description=(.+?)(?:\s+(?:Extension|General\ Subject|Specific\ subject|Topic)=|$)', meta_val)
    if m:
        desc = m.group(1).replace('\\ ', ' ').strip()
        desc = re.sub(r'^Happiness\s+Score\s+explained\s+by\s+', '', desc)
        desc = re.sub(r'^Happiness\s+explained\s+by\s+', '', desc)
        hap_pretty[col] = direct_map_hap.get(desc, desc)

def hap_label(col):
    return hap_pretty.get(col, col)

def hap_group(col):
    if col == hap_target: return 'Outcome'
    if col.startswith('EXP.'): return 'Survey / Experience'
    if col[:3] in ('NY.', 'NV.', 'PA.', 'EG.'): return 'Economy'
    if col[:3] in ('SP.', 'EN.'): return 'Population / Health'
    return 'Other'

# Prepare features
X_hap = df_hap.drop(columns=hap_id_cols + [hap_target], errors='ignore').apply(pd.to_numeric, errors='coerce')
X_hap = X_hap.loc[:, (X_hap.nunique() > 1) & (X_hap.isna().mean() <= 0.35)]
y_hap = pd.to_numeric(df_hap[hap_target], errors='coerce')

print(f'Happiness: {len(df_hap)} rows, {X_hap.shape[1]} features')
print(f'Score range: {y_hap.min():.2f} – {y_hap.max():.2f}')

Happiness: 141 rows, 158 features
Score range: 2.86 – 7.67


### 1.2 Feature → Outcome: Scatter Plots

Each plot shows one feature (x-axis) vs the Happiness Score (y-axis). The red trend line shows the overall direction. 

**What to look for**: GDP, Social Support, and Life Expectancy show strong upward trends — richer, healthier, more supported countries tend to be happier. Generosity and Corruption Perception are weaker. Mortality and Birth Rate go in the opposite direction.

In [ ]:
# Select 9 key features to plot (6 survey + 3 World Bank indicators)
scatter_features = [
    'EXP.GDP.PER.CAPITA', 'EXP.SOC.SUPP', 'EXP.HEAL.LIFE',
    'EXP.FREE.LIFE', 'EXP.GEN', 'EXP.TRUST',
    'SP.DYN.LE00.IN', 'SP.DYN.AMRT.FE', 'SP.DYN.CBRT.IN',
]
scatter_features = [f for f in scatter_features if f in X_hap.columns]

n_cols = 3
n_rows = (len(scatter_features) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(scatter_features):
    mask = X_hap[feat].notna() & y_hap.notna()
    # Scatter: each dot is one country
    axes[i].scatter(X_hap.loc[mask, feat], y_hap[mask],
                    alpha=0.55, s=30, color='#4C78A8', edgecolors='white', linewidth=0.4)
    # Trend line: linear fit to show the overall direction
    z = np.polyfit(X_hap.loc[mask, feat], y_hap[mask], 1)
    xs = np.linspace(X_hap.loc[mask, feat].min(), X_hap.loc[mask, feat].max(), 100)
    axes[i].plot(xs, np.polyval(z, xs), color='#C0392B', linewidth=2, alpha=0.8)
    axes[i].set_title(hap_label(feat), fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Happiness Score' if i % n_cols == 0 else '')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Key Features vs Happiness Score', fontsize=15, fontweight='bold')
fig.tight_layout()
plt.show()

### 1.3 Relation Network

The scatter plots showed us which features matter, one at a time. But they can't show how the features relate to **each other** — for example, GDP and Life Expectancy are strongly correlated (rich countries are healthier). The relation network puts both kinds of relationships in one picture:

- **Solid edges** = feature → outcome (thicker = stronger)
- **Dashed edges** = feature ↔ feature (only shown when correlation is strong)
- **Right side** = positive correlation with happiness; **Left side** = negative

In [ ]:
# Compute Spearman correlations for all features vs Happiness Score
hap_corr = compute_spearman(X_hap, y_hap)
hap_corr['label'] = hap_corr['feature'].map(hap_label)

# Core features we know matter (from the scatter plots above)
hap_core = ['EXP.GDP.PER.CAPITA', 'EXP.SOC.SUPP', 'EXP.HEAL.LIFE',
            'EXP.FREE.LIFE', 'EXP.GEN', 'EXP.TRUST']

# Select a focused subset: core + a few more positives + some negatives
hap_focus = select_focus_features(hap_corr, hap_core, n_extra_pos=4, n_neg=6)
hap_focus_assoc = hap_corr[hap_corr['feature'].isin(hap_focus)].copy()

# Node colors by feature domain
hap_colors = {
    'Outcome': '#F4C542', 'Survey / Experience': '#B07AA1',
    'Economy': '#59A14F', 'Population / Health': '#E15759', 'Other': '#9C755F',
}

draw_relation_network(
    hap_target, 'Happiness Score', hap_focus, hap_focus_assoc,
    hap_label, hap_group, hap_colors, X_hap, ff_threshold=0.40,
    pos_label='Higher Happiness', neg_label='Lower Happiness',
    title='Relation Network — World Happiness',
)

### 1.4 Modeling: Predicting Happiness with XGBoost

The EDA showed which features correlate with happiness. Now let's see if we can **predict** the happiness score from those features using gradient boosting (XGBoost).

- **Task**: Regression (predict a continuous score)
- **Model**: XGBRegressor — builds many small decision trees sequentially, each one correcting the errors of the previous ones
- **Metrics**: RMSE (root mean squared error), MAE (mean absolute error), R² (how much variance is explained)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

# ── Prepare data ──
# Use the survey features (EXP.*) which are complete and most interpretable
hap_model_feats = [c for c in X_hap.columns if c.startswith('EXP.')]
X_hap_model = X_hap[hap_model_feats].copy()

# Impute missing values with the column median (a few countries have gaps)
imputer = SimpleImputer(strategy='median')
X_hap_imp = pd.DataFrame(imputer.fit_transform(X_hap_model),
                         columns=hap_model_feats, index=X_hap_model.index)

# 80/20 train-test split
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_hap_imp, y_hap, test_size=0.2, random_state=42)

# ── Train XGBoost ──
xgb_model = XGBRegressor(
    n_estimators=200,      # number of boosting rounds
    max_depth=4,           # depth of each tree (keep small to avoid overfitting)
    learning_rate=0.1,     # step size shrinkage
    random_state=42,
    verbosity=0,
)
xgb_model.fit(X_train_h, y_train_h)

# ── Evaluate ──
y_pred_h = xgb_model.predict(X_test_h)
rmse = np.sqrt(mean_squared_error(y_test_h, y_pred_h))
mae = mean_absolute_error(y_test_h, y_pred_h)
r2 = r2_score(y_test_h, y_pred_h)

print(f'XGBoost Regression — Happiness Score Prediction')
print(f'  RMSE : {rmse:.3f}  (average error in score units)')
print(f'  MAE  : {mae:.3f}')
print(f'  R²   : {r2:.3f}  (1.0 = perfect, 0.0 = no better than mean)')

# ── Feature importance ──
importances = pd.Series(xgb_model.feature_importances_, index=hap_model_feats)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
importances.plot.barh(ax=ax, color='#4C78A8', edgecolor='white')
ax.set_title('XGBoost Feature Importance — Happiness', fontweight='bold')
ax.set_xlabel('Importance (gain)')
for label in ax.get_yticklabels():
    label.set_text(hap_label(label.get_text()))
ax.set_yticklabels([hap_label(t.get_text()) for t in ax.get_yticklabels()])
fig.tight_layout()
plt.show()

---
# Part 2: Lung Cancer Dataset

**Target**: Lung Cancer (Yes / No)

This dataset contains 800 patients with features about demographics, lifestyle, diet, smoking history, and alcohol history. Unlike the happiness dataset (continuous target), this is a **binary classification** problem.

The strongest signal comes from smoking — so **box plots** comparing the smoking feature values between cancer and non-cancer patients tell the clearest story.

### 2.1 Load Data

In [28]:
df_lc = pd.read_csv('lung_cancer_mixed_types_attributes/LungCancer_cancerYesNo-grouped.csv')
lc_target = 'Class'

lc_pretty = {
    'Class': 'Lung Cancer', 'pd_Sex': 'Sex', 'pd_Age': 'Age',
    'pd_MaritalStatus': 'Marital Status', 'pd_Education': 'Education',
    'pd_Occupation': 'Occupation', 'pd_Height': 'Height', 'pd_Weight': 'Weight',
    'pd_BloodType': 'Blood Type',
    'ls_HousingType': 'Housing Type', 'ls_FloorType': 'Floor Type',
    'ls_FuelType': 'Fuel Type', 'ls_SootType': 'Soot Type',
    'ls_WaterType': 'Water Type', 'ls_SenseOfTaste': 'Sense of Taste',
    'ls_Breakfast': 'Breakfast Habit', 'ls_PhysicalExercise': 'Physical Exercise',
    'fi_Rice(Kg/Month)': 'Rice Intake', 'fi_Noodle(Kg/Month)': 'Noodle Intake',
    'fi_Corn(Kg/Month)': 'Corn Intake', 'fi_FoodGain(Kg/Month)': 'Food Gain',
    'fi_PeanutOil(Kg/Month)': 'Peanut Oil', 'fi_AnimalOil(Kg/Month)': 'Animal Oil',
    'fi_VegetableOil(Kg/Month)': 'Vegetable Oil',
    'fi_FriedFood': 'Fried Food', 'fi_FiredFood': 'Fired Food',
    'fi_RoastedFood': 'Roasted Food', 'fi_PungentFood': 'Pungent Food',
    'fi_SourFood': 'Sour Food', 'fi_GarlicFood': 'Garlic Food',
    'fi_OnionFood': 'Onion Food',
    'sh_Smoking': 'Smoking Status', 'sh_SmokedAge': 'Smoking Start Age',
    'sh_TotalSmoked(Year)': 'Total Years Smoked', 'sh_SmokeEachTime': 'Smoking Intensity',
    'sh_GiveUpSmoking(Age)': 'Quit Smoking Age',
    'sh_SmokerNumber(InHome)': 'Smokers at Home',
    'sh_SmokerNumber(InOffice)': 'Smokers at Work',
    'ah_Drinking': 'Drinking Status', 'ah_DrankAge': 'Drinking Start Age',
    'ah_TotalDrank(Week)': 'Drinks per Week',
    'ah_GiveUpDrinking(Age)': 'Quit Drinking Age',
}

def lc_label(col): return lc_pretty.get(col, col)

def lc_group(col):
    if col == lc_target: return 'Outcome'
    if col.startswith('pd_'): return 'Personal / Demographic'
    if col.startswith('ls_'): return 'Lifestyle / Living'
    if col.startswith('fi_'): return 'Food Intake'
    if col.startswith('sh_'): return 'Smoking History'
    if col.startswith('ah_'): return 'Alcohol History'
    return 'Other'

X_lc = df_lc.drop(columns=[lc_target])
y_lc = df_lc[lc_target].map({'No': 0, 'Yes': 1})

print(f'Lung Cancer: {len(df_lc)} rows, {X_lc.shape[1]} features')
print(f'Class balance: {df_lc[lc_target].value_counts().to_dict()}')

Lung Cancer: 800 rows, 41 features
Class balance: {'No': 450, 'Yes': 350}


### 2.2 Feature → Outcome: Smoking Features by Cancer Status

Each box plot compares the distribution of one smoking feature between cancer patients (red) and non-cancer patients (green).

**What to look for**: If the two boxes are clearly separated (different medians, little overlap), that feature is a strong indicator. Total Years Smoked and Smoking Intensity should show the biggest gap.

In [ ]:
# All 7 smoking-related features
smoking_features = [
    'sh_Smoking', 'sh_TotalSmoked(Year)', 'sh_SmokeEachTime',
    'sh_SmokedAge', 'sh_GiveUpSmoking(Age)',
    'sh_SmokerNumber(InHome)', 'sh_SmokerNumber(InOffice)',
]

n_cols = 4
n_rows = (len(smoking_features) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(smoking_features):
    # Box plot: one box per class (No / Yes), showing median, quartiles, and outliers
    sns.boxplot(data=df_lc, x=lc_target, y=feat, ax=axes[i],
                palette={'No': '#2E8B57', 'Yes': '#C0392B'}, fliersize=3, width=0.5)
    axes[i].set_title(lc_label(feat), fontsize=11, fontweight='bold')
    axes[i].set_xlabel(''); axes[i].set_ylabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Smoking Features — Cancer vs Non-Cancer', fontsize=15, fontweight='bold')
fig.tight_layout()
plt.show()

### 2.3 Feature Ranking: Top Correlations with Lung Cancer

The box plots showed smoking matters, but how does it compare to **all** features? This bar chart ranks every feature by Spearman ρ with the cancer outcome. Red bars = higher value → more cancer risk; green bars = higher value → less cancer risk.

In [ ]:
# Compute Spearman correlations for all features vs Lung Cancer
lc_corr = compute_spearman(X_lc, y_lc)
lc_corr['label'] = lc_corr['feature'].map(lc_label)

# Take the top 15 strongest correlations, sorted for plotting
top = lc_corr.head(15).sort_values('spearman_rho')
colors = top['spearman_rho'].apply(lambda v: '#C0392B' if v >= 0 else '#2E8B57')

fig, ax = plt.subplots(figsize=(10, 6.5))
ax.barh(top['label'], top['spearman_rho'], color=colors, edgecolor='white', linewidth=0.8)
ax.axvline(0, color='#2f2f2f', linewidth=1.2)
margin = max(0.06, top['abs_rho'].max() * 0.18)
ax.set_xlim(top['spearman_rho'].min() - margin, top['spearman_rho'].max() + margin)
ax.set_title('Top 15 Feature Correlations with Lung Cancer')
ax.set_xlabel('Spearman ρ (positive = higher cancer risk)')
ax.grid(axis='x', alpha=0.25, linestyle=':'); ax.grid(axis='y', visible=False)
sns.despine(ax=ax, left=True, bottom=True)

# Print ρ values next to each bar
for patch, val in zip(ax.patches, top['spearman_rho']):
    yp = patch.get_y() + patch.get_height() / 2
    xp = patch.get_width()
    ax.text(xp + (0.005 if val >= 0 else -0.005), yp,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)

fig.tight_layout()
plt.show()

### 2.4 Feature ↔ Feature: Focused Heatmap

The bar chart shows feature→outcome. But do the top features also correlate with **each other**? This heatmap shows pairwise Spearman ρ for the top 15 features + outcome. Look for clusters of strong correlation (dark red/blue blocks) — these are feature groups that move together.

In [ ]:
# Build a correlation matrix for the top 15 features + the outcome
lc_top_feats = lc_corr.head(15)['feature'].tolist()
hm_lc = df_lc[lc_top_feats].copy()
hm_lc['Lung Cancer (Yes=1)'] = y_lc
hm_lc = hm_lc.rename(columns={c: lc_label(c) for c in lc_top_feats})

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(hm_lc.corr(method='spearman'), cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 8.5}, square=True,
            linewidths=0.5, linecolor='white', cbar_kws={'shrink': 0.7}, ax=ax)
ax.set_title('Feature ↔ Feature Correlations (Top 15 + Outcome)', fontsize=13)
ax.tick_params(axis='both', labelsize=9)
fig.tight_layout()
plt.show()

### 2.5 Relation Network

Now we combine the box plots, bar chart, and heatmap into one picture. Notice how the smoking features (red nodes) cluster together on one side — they're all strongly correlated with each other AND with the outcome. The relation network makes this structure immediately visible.

Here `sign_flip=True` because positive ρ means *higher cancer risk* (bad), so we color those edges red instead of green.

In [ ]:
# Core features we expect to matter from domain knowledge
lc_core = ['pd_Age', 'sh_Smoking', 'sh_TotalSmoked(Year)', 'sh_SmokeEachTime',
           'ah_Drinking', 'ls_PhysicalExercise']

# Select focused subset for the network
lc_focus = select_focus_features(lc_corr, lc_core, n_extra_pos=4, n_neg=6)
lc_focus_assoc = lc_corr[lc_corr['feature'].isin(lc_focus)].copy()

# Node colors — one color per feature domain group
lc_colors = {
    'Outcome': '#F4C542', 'Personal / Demographic': '#4C78A8',
    'Lifestyle / Living': '#59A14F', 'Food Intake': '#B07AA1',
    'Smoking History': '#E15759', 'Alcohol History': '#9C755F', 'Other': '#76B7B2',
}

draw_relation_network(
    lc_target, 'Lung Cancer (Yes)', lc_focus, lc_focus_assoc,
    lc_label, lc_group, lc_colors, X_lc, ff_threshold=0.22,
    pos_label='Higher Cancer Risk', neg_label='Lower Cancer Risk',
    sign_flip=True,  # positive ρ = bad (more cancer), so flip red/green
    title='Relation Network — Lung Cancer',
)

### 2.6 Modeling: Classifying Lung Cancer with a Decision Tree

The EDA showed smoking features dominate the correlation structure. A **decision tree** is a natural fit here — it learns a series of if/then splits (e.g., "if Total Years Smoked > 15 → Cancer likely") that are easy to visualize and explain.

- **Task**: Binary classification (Cancer Yes / No)
- **Model**: DecisionTreeClassifier (max depth = 5 to keep the tree readable)
- **Metrics**: Accuracy, Precision, Recall, F1-score, Confusion Matrix

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# ── Prepare data ──
# Encode categorical columns as integers (Decision Trees handle this natively)
X_lc_model = X_lc.copy()
label_encoders = {}
for col in X_lc_model.columns:
    if X_lc_model[col].dtype == 'object':
        le = LabelEncoder()
        X_lc_model[col] = le.fit_transform(X_lc_model[col].astype(str))
        label_encoders[col] = le

# 80/20 stratified split (keeps the Yes/No ratio balanced in both sets)
X_train_lc, X_test_lc, y_train_lc, y_test_lc = train_test_split(
    X_lc_model, y_lc, test_size=0.2, random_state=42, stratify=y_lc)

# ── Train Decision Tree ──
dt_model = DecisionTreeClassifier(
    max_depth=5,           # keep tree shallow enough to visualize
    random_state=42,
    class_weight='balanced',  # handle slight class imbalance
)
dt_model.fit(X_train_lc, y_train_lc)

# ── Evaluate ──
y_pred_lc = dt_model.predict(X_test_lc)
acc = accuracy_score(y_test_lc, y_pred_lc)

print(f'Decision Tree — Lung Cancer Classification')
print(f'  Accuracy: {acc:.3f}\n')
print(classification_report(y_test_lc, y_pred_lc, target_names=['No Cancer', 'Cancer']))

# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test_lc, y_pred_lc, display_labels=['No Cancer', 'Cancer'],
    cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix', fontweight='bold')

# ── Tree visualization ──
# Show the top 3 levels of the tree — each node is a split decision
plot_tree(dt_model, max_depth=3, filled=True, rounded=True,
          feature_names=[lc_label(c) for c in X_lc_model.columns],
          class_names=['No Cancer', 'Cancer'],
          fontsize=7, ax=axes[1])
axes[1].set_title('Decision Tree (top 3 levels)', fontweight='bold')

fig.tight_layout()
plt.show()

---
# Part 3: Network Intrusion Detection (NSL-KDD)

**Target**: Attack vs Normal traffic (binary)

This dataset contains ~125K network connections, each described by 41 features extracted from TCP/IP packet headers and traffic patterns. The features are organized into 4 domain groups:

| Group | Features | What they capture |
|---|---|---|
| **Basic** (1–9) | duration, src_bytes, dst_bytes, … | Packet header info — what the connection looks like |
| **Content** (10–22) | failed_logins, root_shell, … | Payload analysis — what the connection did |
| **Time-based** (23–31) | count, serror_rate, … | Traffic patterns in a 2-second window |
| **Host-based** (32–41) | dst_host_count, dst_host_serror_rate, … | Traffic patterns over 100 connections |

> Dataset: [NSL-KDD](https://www.kaggle.com/datasets/hassan06/nslkdd) — download `KDDTrain+.txt` and place it in `nsl_kdd/`.

### 3.1 Load Data

The NSL-KDD CSV files have **no header row** — we add column names manually. We convert the multi-class attack labels to binary: `normal` → 0, everything else → 1.

In [ ]:
# Data already downloaded in the Setup cell above
nsl_file = Path('nsl_kdd') / 'KDDTrain+.txt'

# Column names (41 features + label + difficulty)
nsl_columns = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent',
    'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
    'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login',
    'count', 'srv_count',
    'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
    'label', 'difficulty',
]

df_nsl = pd.read_csv(nsl_file, header=None, names=nsl_columns)

# Binary target: normal=0, attack=1
nsl_target = 'is_attack'
df_nsl[nsl_target] = (df_nsl['label'] != 'normal').astype(int)

# Drop non-numeric columns for correlation analysis
nsl_cat_cols = ['protocol_type', 'service', 'flag', 'label', 'difficulty']
X_nsl = df_nsl.drop(columns=nsl_cat_cols + [nsl_target])
X_nsl = X_nsl.loc[:, X_nsl.nunique() > 1]  # drop constant columns
y_nsl = df_nsl[nsl_target]

# Pretty names
nsl_pretty = {
    'duration': 'Duration', 'src_bytes': 'Src Bytes', 'dst_bytes': 'Dst Bytes',
    'land': 'Land', 'wrong_fragment': 'Wrong Fragment', 'urgent': 'Urgent',
    'hot': 'Hot Indicators', 'num_failed_logins': 'Failed Logins',
    'logged_in': 'Logged In', 'num_compromised': 'Compromised',
    'root_shell': 'Root Shell', 'su_attempted': 'SU Attempted',
    'num_root': 'Root Accesses', 'num_file_creations': 'File Creations',
    'num_shells': 'Shells Opened', 'num_access_files': 'Access Files',
    'num_outbound_cmds': 'Outbound Cmds', 'is_host_login': 'Host Login',
    'is_guest_login': 'Guest Login',
    'count': 'Count (2s)', 'srv_count': 'Srv Count (2s)',
    'serror_rate': 'SYN Error Rate', 'srv_serror_rate': 'Srv SYN Error Rate',
    'rerror_rate': 'REJ Error Rate', 'srv_rerror_rate': 'Srv REJ Error Rate',
    'same_srv_rate': 'Same Srv Rate', 'diff_srv_rate': 'Diff Srv Rate',
    'srv_diff_host_rate': 'Srv Diff Host Rate',
    'dst_host_count': 'Dst Host Count', 'dst_host_srv_count': 'Dst Host Srv Count',
    'dst_host_same_srv_rate': 'Dst Same Srv Rate',
    'dst_host_diff_srv_rate': 'Dst Diff Srv Rate',
    'dst_host_same_src_port_rate': 'Dst Same Port Rate',
    'dst_host_srv_diff_host_rate': 'Dst Srv Diff Host',
    'dst_host_serror_rate': 'Dst SYN Error Rate',
    'dst_host_srv_serror_rate': 'Dst Srv SYN Error',
    'dst_host_rerror_rate': 'Dst REJ Error Rate',
    'dst_host_srv_rerror_rate': 'Dst Srv REJ Error',
}
def nsl_label(col): return nsl_pretty.get(col, col)

# Feature groups (4 standard NSL-KDD categories)
_basic = {'duration', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent'}
_content = {'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
            'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
            'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login'}
_time = {'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate',
         'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate'}
_host = {'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
         'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
         'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
         'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate'}

def nsl_group(col):
    if col == nsl_target: return 'Outcome'
    if col in _basic: return 'Basic (Packet)'
    if col in _content: return 'Content (Payload)'
    if col in _time: return 'Time-based Traffic'
    if col in _host: return 'Host-based Traffic'
    return 'Other'

print(f'NSL-KDD: {len(df_nsl):,} rows, {X_nsl.shape[1]} numeric features')
print(f'Class balance: Normal={( y_nsl==0).sum():,}  Attack={(y_nsl==1).sum():,}')
print(f'Attack types: {df_nsl["label"].nunique()} unique labels')

### 3.2 Feature → Outcome: Key Traffic Features by Attack Status

We pick the top 2 most discriminative features from each of the 4 groups and show box plots comparing attack vs normal traffic. This reveals which groups carry the most signal — and whether the signal comes from one group or is spread across all four.

In [ ]:
# Compute correlations for all numeric features vs attack/normal
nsl_corr = compute_spearman(X_nsl, y_nsl)
nsl_corr['label'] = nsl_corr['feature'].map(nsl_label)
nsl_corr['group'] = nsl_corr['feature'].map(nsl_group)

# Pick top 2 features from each of the 4 groups
box_feats = []
for grp in ['Basic (Packet)', 'Content (Payload)', 'Time-based Traffic', 'Host-based Traffic']:
    sub = nsl_corr[nsl_corr['group'] == grp].head(2)['feature'].tolist()
    box_feats.extend(sub)

# Add a readable label column for plotting
df_nsl['Attack Status'] = df_nsl[nsl_target].map({0: 'Normal', 1: 'Attack'})

n_cols = 4
n_rows = (len(box_feats) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(box_feats):
    sns.boxplot(data=df_nsl, x='Attack Status', y=feat, ax=axes[i],
                palette={'Normal': '#2E8B57', 'Attack': '#C0392B'},
                fliersize=1.5, width=0.5)
    grp = nsl_group(feat)
    axes[i].set_title(f'{nsl_label(feat)}\n({grp})', fontsize=10, fontweight='bold')
    axes[i].set_xlabel(''); axes[i].set_ylabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Top Features per Group — Attack vs Normal', fontsize=15, fontweight='bold')
fig.tight_layout()
plt.show()

### 3.3 Multi-Feature View: Parallel Coordinates

Box plots show one feature at a time. But attacks have a **multi-feature signature** — certain combinations of values that appear together. Parallel coordinates show every sampled connection as a line crossing all features, colored by class. Where the red and green bands separate, that feature helps distinguish attacks from normal traffic.

In [ ]:
# Pick top 3 features per group (12 total) for the parallel coordinates
pc_feats = []
for grp in ['Basic (Packet)', 'Content (Payload)', 'Time-based Traffic', 'Host-based Traffic']:
    sub = nsl_corr[nsl_corr['group'] == grp].head(3)['feature'].tolist()
    pc_feats.extend(sub)

# Subsample 600 connections (300 normal + 300 attack) — too many lines = solid blob
np.random.seed(42)
n_sample = 600
idx_norm = df_nsl[df_nsl[nsl_target] == 0].sample(n_sample // 2, random_state=42).index
idx_atk  = df_nsl[df_nsl[nsl_target] == 1].sample(n_sample // 2, random_state=42).index
sample = df_nsl.loc[idx_norm.union(idx_atk)].copy()

# Normalize each feature to [0, 1] so all axes are comparable
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
sample_scaled = pd.DataFrame(
    scaler.fit_transform(sample[pc_feats]),
    columns=pc_feats, index=sample.index,
)
sample_scaled[nsl_target] = sample[nsl_target].values

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(pc_feats))

# Draw each connection as a semi-transparent line across all features
for is_atk, color, alpha, label in [(0, '#2E8B57', 0.08, 'Normal'),
                                     (1, '#C0392B', 0.08, 'Attack')]:
    subset = sample_scaled[sample_scaled[nsl_target] == is_atk]
    for _, row in subset.iterrows():
        ax.plot(x, row[pc_feats].values, color=color, alpha=alpha, linewidth=0.6)
    # One opaque line for the legend
    ax.plot([], [], color=color, alpha=0.8, linewidth=2, label=label)

ax.set_xticks(x)
ax.set_xticklabels([nsl_label(f) for f in pc_feats], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Normalized Value (0–1)')
ax.set_title('Parallel Coordinates — Attack vs Normal Traffic Signatures', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')

# Add vertical separators between feature groups
cum = 0
for grp in ['Basic (Packet)', 'Content (Payload)', 'Time-based Traffic', 'Host-based Traffic']:
    n_in_grp = len(nsl_corr[nsl_corr['group'] == grp].head(3))
    if cum > 0:
        ax.axvline(cum - 0.5, color='#aaa', linewidth=1, linestyle='--', alpha=0.6)
    mid = cum + n_in_grp / 2 - 0.5
    ax.text(mid, 1.08, grp, ha='center', fontsize=8.5, fontstyle='italic',
            transform=ax.get_xaxis_transform())
    cum += n_in_grp

ax.set_xlim(-0.5, len(pc_feats) - 0.5)
fig.tight_layout()
plt.show()

### 3.4 Multi-Feature View: Radar Chart

Another way to compare attack vs normal: overlay the **median feature profile** of each class. The two shapes immediately show where they diverge. Features where the red shape sticks out much further (or much less) than the green shape are the most discriminative.

In [ ]:
# Same features as the parallel coordinates
radar_feats = pc_feats

# Compute the median value per class, then normalize to [0, 1]
medians = df_nsl.groupby(nsl_target)[radar_feats].median()
mins = df_nsl[radar_feats].min()
maxs = df_nsl[radar_feats].max()
ranges = maxs - mins
ranges[ranges == 0] = 1  # avoid division by zero
medians_norm = (medians - mins) / ranges

labels = [nsl_label(f) for f in radar_feats]
n = len(radar_feats)

# Radar chart: arrange features evenly around a circle
angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

for cls, color, lbl in [(0, '#2E8B57', 'Normal'), (1, '#C0392B', 'Attack')]:
    values = medians_norm.loc[cls].tolist()
    values += values[:1]  # close the polygon
    ax.plot(angles, values, color=color, linewidth=2.5, label=lbl)
    ax.fill(angles, values, color=color, alpha=0.12)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75])
ax.set_yticklabels(['0.25', '0.50', '0.75'], fontsize=8, color='#888')
ax.set_title('Attack vs Normal — Median Feature Profiles (Normalized)',
             fontsize=14, fontweight='bold', pad=24)
ax.legend(fontsize=12, loc='upper right', bbox_to_anchor=(1.15, 1.1))

# Add group labels around the outside
cum = 0
for grp in ['Basic (Packet)', 'Content (Payload)', 'Time-based Traffic', 'Host-based Traffic']:
    n_in_grp = len(nsl_corr[nsl_corr['group'] == grp].head(3))
    mid_idx = cum + n_in_grp // 2
    mid_angle = angles[mid_idx]
    ax.text(mid_angle, 1.25, grp, ha='center', va='center',
            fontsize=8.5, fontstyle='italic', color='#555')
    cum += n_in_grp

fig.tight_layout()
plt.show()

### 3.5 Relation Network

Finally, the relation network combines everything: which features drive attack detection, how the 4 domain groups connect internally, and which features work together. The 4-color grouping (blue = Basic, green = Content, red = Time, purple = Host) makes the domain structure immediately visible.

In [ ]:
# Core features — one from each group, plus a few known to be important
nsl_core = ['src_bytes', 'logged_in', 'count', 'serror_rate',
            'same_srv_rate', 'dst_host_srv_count', 'dst_host_same_srv_rate']

# Select focused subset for the network
nsl_focus = select_focus_features(nsl_corr, nsl_core, n_extra_pos=5, n_neg=5)
nsl_focus_assoc = nsl_corr[nsl_corr['feature'].isin(nsl_focus)].copy()

# One color per feature group — matches the 4 NSL-KDD categories
nsl_colors = {
    'Outcome': '#F4C542',
    'Basic (Packet)': '#4C78A8',
    'Content (Payload)': '#59A14F',
    'Time-based Traffic': '#E15759',
    'Host-based Traffic': '#B07AA1',
}

draw_relation_network(
    nsl_target, 'Attack', nsl_focus, nsl_focus_assoc,
    nsl_label, nsl_group, nsl_colors, X_nsl, ff_threshold=0.35,
    pos_label='More Attack-like', neg_label='More Normal-like',
    sign_flip=True,  # positive ρ = more attack-like, so flip red/green
    title='Relation Network — Network Intrusion (NSL-KDD)',
)

### 3.6 Modeling: Detecting Intrusions with SVM

Support Vector Machines (SVM) find a decision boundary that maximizes the margin between classes. The RBF kernel lets it learn non-linear boundaries — ideal for the complex feature interactions in network traffic.

- **Task**: Binary classification (Attack / Normal)
- **Model**: SVC with RBF kernel
- **Note**: SVM scales ~O(n²) with sample size. We **subsample to 20K rows** (10K per class) so training finishes in under a minute. For full-dataset speed, consider `LinearSVC` instead.

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# ── Subsample for SVM speed ──
# 10K normal + 10K attack = 20K total (full 125K would take 10+ min)
np.random.seed(42)
n_per_class = 10_000
idx_norm_svm = df_nsl[df_nsl[nsl_target] == 0].sample(n_per_class, random_state=42).index
idx_atk_svm  = df_nsl[df_nsl[nsl_target] == 1].sample(n_per_class, random_state=42).index
idx_svm = idx_norm_svm.union(idx_atk_svm)

X_nsl_sub = X_nsl.loc[idx_svm]
y_nsl_sub = y_nsl.loc[idx_svm]

print(f'Subsampled: {len(X_nsl_sub):,} rows ({n_per_class:,} per class)')

# 80/20 stratified split
X_train_nsl, X_test_nsl, y_train_nsl, y_test_nsl = train_test_split(
    X_nsl_sub, y_nsl_sub, test_size=0.2, random_state=42, stratify=y_nsl_sub)

# ── Scale features ──
# SVM is sensitive to feature magnitudes — StandardScaler makes each feature
# have mean=0, std=1, so no single feature dominates the distance calculation.
scaler_nsl = StandardScaler()
X_train_scaled = scaler_nsl.fit_transform(X_train_nsl)
X_test_scaled  = scaler_nsl.transform(X_test_nsl)

# ── Train SVM ──
print('Training SVM (this may take ~30–60 seconds) …')
svm_model = SVC(kernel='rbf', C=1.0, random_state=42)
svm_model.fit(X_train_scaled, y_train_nsl)
print('Done.')

# ── Evaluate ──
y_pred_nsl = svm_model.predict(X_test_scaled)
acc_nsl = accuracy_score(y_test_nsl, y_pred_nsl)

print(f'\nSVM (RBF) — Network Intrusion Detection')
print(f'  Accuracy: {acc_nsl:.3f}\n')
print(classification_report(y_test_nsl, y_pred_nsl, target_names=['Normal', 'Attack']))

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test_nsl, y_pred_nsl, display_labels=['Normal', 'Attack'],
    cmap='Blues', ax=ax)
ax.set_title('SVM Confusion Matrix — NSL-KDD', fontweight='bold')
fig.tight_layout()
plt.show()

---
# Summary

| Dataset | Hero Visualization | Relation Network Shows | Model | Task |
|---|---|---|---|---|
| World Happiness | Scatter plots | GDP, social support, health cluster together as happiness drivers | XGBoost Regressor | Regression |
| Lung Cancer | Smoking box plots | Smoking features dominate and are tightly inter-correlated | Decision Tree | Classification |
| Network Intrusion | Grouped box plots | 4 domain groups contribute different attack signals | SVM (RBF kernel) | Classification |

Each part follows: **hero visualization → relation network → model**. The EDA visualizations help us understand *what drives the outcome and how drivers connect*. The model then turns that understanding into a *prediction* — can we actually use these features to classify or predict the target?

**To use this notebook on Colab**: push the data files to a GitHub repo and update `DATA_BASE_URL` in the first code cell.